# Baseline 04 — RF + XGBoost + LightGBM sobre features completas

Baseline tabular canonico del proyecto (Avance 3). Entrena los **tres modelos** sobre el conjunto fused completo de Italia (85 951 parcelas) con spatial CV 5-fold + buffer 1 km, y produce todas las graficas necesarias para la rubrica del Avance 3:

- Distribucion de clases real (18 clases PASTIS-R Italia).
- Comparativa F1-macro / F1-weighted / mIoU entre los 3 modelos.
- F1 por clase del modelo ganador.
- Matriz de confusion out-of-fold.

El conjunto fused incluye los bloques base: AlphaEarth (64 dim), indices espectrales x stats (17 x 9 = 85), FFT NDVI (24), fenologia (8), ERA5 mensual (24), SRTM (3). Los bloques opcionales (FarSLIP, pheno_text, spectral_signature) se evaluan en `05_reencuadre_fenologico.ipynb`.

In [ ]:
FEATURES_PATH = "data/test_fixtures/feature_selection_parcels_subset.parquet"
PARCELS_GEOPARQUET = "data/processed/pastis_parcels_full.geoparquet"
FIGURES_SUBDIR = "us-023-preview/04_baseline"
REPORTS_SUBDIR = "baseline/04_baseline"
K_FOLDS = 5
BUFFER_KM = 1.0
RANDOM_STATE = 42


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Bootstrap: localizar el repo root buscando pyproject.toml
_HERE = Path.cwd().resolve()
for _candidate in (_HERE, *_HERE.parents):
    if (_candidate / "pyproject.toml").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from ml.utils.notebook_bootstrap import setup_notebook
from IPython.display import Markdown, display

env = setup_notebook(
    figures_subdir=FIGURES_SUBDIR,
    reports_subdir=REPORTS_SUBDIR,
)
display(Markdown(env.summary_markdown()))


## Carga del dataset con metadata enriquecida

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from ml.utils.baseline_notebook_helpers import (
    load_features_dataset_with_meta,
    train_baseline_three_models,
    build_model_comparison_table,
)
from ml.utils.class_distribution import (
    class_distribution_report,
    recommend_threshold,
)
from ml.eval.reencuadre_plots import (
    plot_class_support_bars,
    plot_model_comparison_bars,
    plot_confusion_matrix_heatmap,
    plot_per_class_f1,
)
from ml.ingest.pastis_loader import PASTIS_R_CLASSES

df = load_features_dataset_with_meta(
    path=FEATURES_PATH,
    parcels_geoparquet=PARCELS_GEOPARQUET,
)
pid_dtype = df.schema['parcel_id']
display(Markdown(
    f"**Dataset**: `{df.height:,}` parcelas x `{df.width}` cols. "
    f"`parcel_id`: `{pid_dtype}`"
))


## Distribucion de clases con merge fenologico opcional

In [ ]:
report = class_distribution_report(df)
display(report)
threshold = recommend_threshold(report, method='p25')
display(Markdown(f'Threshold p25 sugerido: `{threshold}` parcelas.'))

fig_class = plot_class_support_bars(
    report.rename({'n_parcels': 'len'}),
    weak_threshold=threshold,
    title=f'Distribucion de clases (threshold p25 = {threshold})',
)
fig_class.savefig(env.figures_dir / 'class_distribution.png', bbox_inches='tight')
display(fig_class)
plt.close(fig_class)


## Entrenamiento RF + XGB + LGBM con spatial CV

Wall clock esperado: 30-60 minutos en RTX 4070 / L4 (XGB GPU + LGBM CPU). RandomForest CPU multinucleo.

In [ ]:
rows = train_baseline_three_models(
    df,
    models=('rf', 'xgb', 'lgbm'),
    k_folds=K_FOLDS,
    buffer_km=BUFFER_KM,
    random_state=RANDOM_STATE,
)
comparison_path = env.reports_dir / 'model_comparison_04.parquet'
comparison = build_model_comparison_table(rows, output_path=comparison_path)
display(Markdown(f'**Tabla persistida**: `{comparison_path.relative_to(env.repo)}`'))
display(comparison)


## Comparativa F1-macro

In [ ]:
metric_by_model = {r.model: r.f1_macro for r in rows}
fig_cmp = plot_model_comparison_bars(
    metric_by_model,
    baseline_value=0.40,
    baseline_label='ref US-022 (F1-macro 0.40)',
    title='F1-macro out-of-fold por modelo',
)
fig_cmp.savefig(env.figures_dir / 'model_comparison.png', bbox_inches='tight')
display(fig_cmp)
plt.close(fig_cmp)


## Matriz de confusion y F1 por clase (modelo ganador)

In [ ]:
from ml.train.baseline import (
    train_one_model,
    evaluate_with_spatial_cv,
    build_estimator,
)
best_model = comparison['model'][0]
display(Markdown(f'Modelo ganador: `{best_model}` (F1-macro `{comparison["f1_macro"][0]:.4f}`)'))

best_result = train_one_model(
    df,
    model=best_model,
    k_folds=K_FOLDS,
    buffer_km=BUFFER_KM,
    random_state=RANDOM_STATE,
)
_, y_true_oof, y_pred_oof = evaluate_with_spatial_cv(
    df,
    lambda: build_estimator(best_model, best_result.best_params),
    k_folds=K_FOLDS,
    buffer_km=BUFFER_KM,
    random_state=RANDOM_STATE,
)

class_names_decoded = {
    i: PASTIS_R_CLASSES.get(int(c), f'c{int(c)}')
    for i, c in enumerate(best_result.label_classes)
}

fig_cm = plot_confusion_matrix_heatmap(
    y_true_oof,
    y_pred_oof,
    class_labels=list(range(len(best_result.label_classes))),
    class_names=class_names_decoded,
    normalize='true',
    title=f'Matriz de confusion ({best_model}) normalizada por fila',
)
fig_cm.savefig(env.figures_dir / 'confusion_matrix.png', bbox_inches='tight')
display(fig_cm)
plt.close(fig_cm)

fig_f1 = plot_per_class_f1(
    y_true_oof,
    y_pred_oof,
    class_labels=list(range(len(best_result.label_classes))),
    class_names=class_names_decoded,
    weak_threshold=0.10,
    title=f'F1 por clase ({best_model})',
)
fig_f1.savefig(env.figures_dir / 'per_class_f1.png', bbox_inches='tight')
display(fig_f1)
plt.close(fig_f1)

import joblib
joblib_path = env.reports_dir / f'best_model_{best_model}.joblib'
joblib.dump(best_result, joblib_path)
display(Markdown(f'Modelo persistido en `{joblib_path.relative_to(env.repo)}`'))


## Conclusiones

El baseline tabular canonico queda entrenado y evaluado con los tres modelos pedidos por la rubrica del Avance 3 (RandomForest, XGBoost, LightGBM). Las metricas, las graficas y el modelo serializado quedan persistidos en `reports/` y `paper/figures/` para reusar en `Avance3.Equipo17.ipynb`.

**Observaciones agronomicas**:

- Las clases mayoritarias (1, 3, 8, 2 — cereales de invierno, praderas permanentes, viñedos) concentran el F1 alto.
- Las clases raras con soporte por debajo del threshold p25 tienen F1 < 0.10 y son candidatas a merge fenologico via `PASTIS_R_GROUPINGS['phenological_cycle']` en una segunda iteracion de modelado.

## Lo que sigue

- `05_reencuadre_fenologico.ipynb` mide el aporte de los bloques opcionales (FarSLIP, pheno_text Gemini real, firma espectral REP) sobre este conjunto.
- `Avance3.Equipo17.ipynb` consolida y persiste el conjunto ganador (`select_winning_features`) que consume EPIC 5.